# 04 รายงาน interactive ของ LABAI

notebook นี้สร้างรายงาน HTML แบบ interactive จากไฟล์ cleaned ของ Phase 3
รายงานสรุปผลเชิงพรรณนาและเชิงวินิจฉัยเท่านั้น ยังไม่มีการสร้าง ML model

In [1]:
from pathlib import Path
from html import escape

import pandas as pd
import plotly.express as px
import plotly.io as pio

## ค้นหา root ของโครงการ

รองรับการเปิด notebook จาก `notebooks/` หรือจาก root ของโครงการ โดยตรวจหา `data/raw/` ที่มีอยู่จริง

In [2]:
def find_project_root():
    current_path = Path.cwd().resolve()
    candidate_paths = [current_path]
    candidate_paths.extend(current_path.parents)

    for candidate_path in candidate_paths:
        if (candidate_path / "data" / "raw").exists():
            return candidate_path

    raise FileNotFoundError("ไม่พบ root ของโครงการที่มี data/raw")


PROJECT_ROOT = find_project_root()

## กำหนด path ของข้อมูลและรายงาน

cell นี้กำหนดตำแหน่งไฟล์ cleaned และตำแหน่งรายงาน HTML ที่จะสร้างจาก root ที่ตรวจพบ

In [3]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
HTML_REPORT_DIR = PROJECT_ROOT / "reports" / "html"
HTML_REPORT_DIR.mkdir(parents=True, exist_ok=True)

DEMAND_SUPPLY_FILE = PROCESSED_DATA_DIR / "labai_demand_supply_cleaned.csv"
MATCHING_FILE = PROCESSED_DATA_DIR / "labai_land_matching_cleaned.csv"
LANDOWNERS_FILE = PROCESSED_DATA_DIR / "labai_landowners_cleaned.csv"
REPORT_FILE = HTML_REPORT_DIR / "labai_phase3_interactive_report.html"
OFFLINE_REPORT_FILE = HTML_REPORT_DIR / "labai_phase3_interactive_report_offline.html"

## โหลดข้อมูล demand/supply ที่ผ่าน cleaning

ไฟล์นี้ใช้ `province_name`, `count_want_land`, `count_owner_land` และ `count_gap_want_minus_owner`

In [4]:
demand_supply_df = pd.read_csv(DEMAND_SUPPLY_FILE, encoding="utf-8-sig")
demand_supply_df.head()

,PROVINCE_NAME,countw,count_want_land,area_rai_want_land,area_nga_want_land,area_wa_want_land,countowner,count_owner_land,area_rai_owner,area_nga_owner,...,province_name,area_rai_want_land_numeric,area_rai_want_land_has_placeholder,area_nga_want_land_numeric,area_nga_want_land_has_placeholder,area_rai_owner_numeric,area_rai_owner_has_placeholder,area_nga_owner_numeric,area_nga_owner_has_placeholder,count_gap_want_minus_owner
0,กรุงเทพมหานคร,164,39,0,2,53.0,36,1,0,2,...,กรุงเทพมหานคร,0,False,2,False,0,False,2,False,38
1,สมุทรปราการ,29,22,0,0,0.0,6,0,0,0,...,สมุทรปราการ,0,False,0,False,0,False,0,False,22
2,นนทบุรี,35,30,0,0,0.0,6,0,0,0,...,นนทบุรี,0,False,0,False,0,False,0,False,30
3,ปทุมธานี,44,72,1,0,0.0,2,1,1,0,...,ปทุมธานี,1,False,0,False,1,False,0,False,71
4,พระนครศรีอยุธยา,15,19,0,0,0.0,3,0,0,0,...,พระนครศรีอยุธยา,0,False,0,False,0,False,0,False,19


## โหลดข้อมูลการจับคู่ที่ผ่าน cleaning

ไฟล์นี้เก็บค่า raw, column `_numeric` และ flag `_has_placeholder` จากการตัดสินใจ cleaning ใน Phase 3

In [5]:
matching_df = pd.read_csv(MATCHING_FILE, encoding="utf-8-sig")
matching_df.head()

,จังหวัด,จับคู่ได้,ผู้ประสงค์,เจ้าของที่ดิน,แปลงที่ดิน,province_name,จับคู่ได้_numeric,จับคู่ได้_has_placeholder,ผู้ประสงค์_numeric,ผู้ประสงค์_has_placeholder,เจ้าของที่ดิน_numeric,เจ้าของที่ดิน_has_placeholder,แปลงที่ดิน_numeric,แปลงที่ดิน_has_placeholder
0,กรุงเทพมหานคร,1,1,1,1,กรุงเทพมหานคร,1.0,False,1.0,False,1.0,False,1.0,False
1,สมุทรปราการ,-,-,-,-,สมุทรปราการ,NaN,True,NaN,True,NaN,True,NaN,True
2,นนทบุรี,-,-,-,-,นนทบุรี,NaN,True,NaN,True,NaN,True,NaN,True
3,ปทุมธานี,1,1,1,1,ปทุมธานี,1.0,False,1.0,False,1.0,False,1.0,False
4,พระนครศรีอยุธยา,-,-,-,-,พระนครศรีอยุธยา,NaN,True,NaN,True,NaN,True,NaN,True


## โหลดข้อมูลเจ้าของที่ดินที่ผ่าน cleaning

ไฟล์นี้ใช้ประกอบการตรวจว่า report โหลด cleaned output ครบทั้งสามชุดข้อมูล

In [6]:
landowners_df = pd.read_csv(LANDOWNERS_FILE, encoding="utf-8-sig")
landowners_df.head()

,PROVINCE_NAME,countowner,count_owner_land,area_rai,area_nga,area_wa,province_name,area_rai_numeric,area_rai_has_placeholder,area_nga_numeric,area_nga_has_placeholder
0,กรุงเทพมหานคร,36,1,0,2,53.0,กรุงเทพมหานคร,0,False,2,False
1,สมุทรปราการ,6,0,0,0,0.0,สมุทรปราการ,0,False,0,False
2,นนทบุรี,6,0,0,0,0.0,นนทบุรี,0,False,0,False
3,ปทุมธานี,2,1,1,0,0.0,ปทุมธานี,1,False,0,False
4,พระนครศรีอยุธยา,3,0,0,0,0.0,พระนครศรีอยุธยา,0,False,0,False


## ระบุ column ที่ตรวจพบจริง

ชื่อ column เหล่านี้มาจาก schema audit และ cleaned outputs ของโครงการ

In [7]:
PROVINCE_COLUMN = "province_name"
WANT_COUNT_COLUMN = "count_want_land"
OWNER_LAND_COUNT_COLUMN = "count_owner_land"
GAP_COLUMN = "count_gap_want_minus_owner"
MATCHED_NUMERIC_COLUMN = "จับคู่ได้_numeric"
MATCHING_PLACEHOLDER_COLUMN = "จับคู่ได้_has_placeholder"
PLOTLY_FONT_FAMILY = "Sarabun, system-ui, sans-serif"
IDEA_GALLERY_TITLE = "ไอเดียต่อยอดจากชุดข้อมูลอื่น"

## คำนวณ metrics สำหรับรายงาน

ค่า gap ใช้สูตร `count_want_land - count_owner_land` ตาม Phase 3 และไม่ยืนยันนิยามเชิงนโยบายเพิ่มเติม

In [8]:
total_want_count = int(demand_supply_df[WANT_COUNT_COLUMN].sum())
total_owner_land_count = int(demand_supply_df[OWNER_LAND_COUNT_COLUMN].sum())
highest_gap_row = demand_supply_df.nlargest(1, GAP_COLUMN).iloc[0]
positive_gap_count = int((demand_supply_df[GAP_COLUMN] > 0).sum())
negative_gap_count = int((demand_supply_df[GAP_COLUMN] < 0).sum())
zero_gap_count = int((demand_supply_df[GAP_COLUMN] == 0).sum())
demand_supply_correlation = demand_supply_df[WANT_COUNT_COLUMN].corr(
    demand_supply_df[OWNER_LAND_COUNT_COLUMN]
)
observed_matching_row_count = int(matching_df[MATCHED_NUMERIC_COLUMN].notna().sum())
matching_placeholder_row_count = int(matching_df[MATCHING_PLACEHOLDER_COLUMN].sum())

## แสดง key metrics

ตารางนี้เป็นข้อมูลเดียวกับที่จะปรากฏบน metric cards ในรายงาน HTML

In [9]:
key_metrics_df = pd.DataFrame(
    [
        {"metric": "ผลรวม count_want_land", "value": total_want_count},
        {"metric": "ผลรวม count_owner_land", "value": total_owner_land_count},
        {
            "metric": "จังหวัดที่มี gap สูงสุด",
            "value": f"{highest_gap_row[PROVINCE_COLUMN]} ({int(highest_gap_row[GAP_COLUMN])})",
        },
        {"metric": "จำนวนจังหวัด gap บวก", "value": positive_gap_count},
        {"metric": "จำนวนจังหวัด gap ลบ", "value": negative_gap_count},
        {"metric": "จำนวนจังหวัด gap เท่ากับศูนย์", "value": zero_gap_count},
        {"metric": "Pearson correlation", "value": round(demand_supply_correlation, 3)},
        {"metric": "แถวการจับคู่ที่มีตัวเลข", "value": observed_matching_row_count},
        {"metric": "แถวที่จับคู่ได้เป็น -", "value": matching_placeholder_row_count},
    ]
)
key_metrics_df

,metric,value
0,ผลรวม count_want_land,1371
1,ผลรวม count_owner_land,80
2,จังหวัดที่มี gap สูงสุด,ลำพูน (308)
3,จำนวนจังหวัด gap บวก,68
4,จำนวนจังหวัด gap ลบ,3
5,จำนวนจังหวัด gap เท่ากับศูนย์,6
6,Pearson correlation,0.466
7,แถวการจับคู่ที่มีตัวเลข,7
8,แถวที่จับคู่ได้เป็น -,70


## กำหนด font สำหรับกราฟ Plotly

ใช้ Sarabun เป็นลำดับแรกทั้งข้อความในกราฟและ hover labels
หากเบราว์เซอร์อ่านไฟล์ font ในโครงการไม่ได้ จะใช้ system font เป็น fallback

In [10]:
def apply_plotly_font(figure):
    figure.update_layout(
        font={"family": PLOTLY_FONT_FAMILY},
        hoverlabel={"font": {"family": PLOTLY_FONT_FAMILY}},
    )
    return figure

## สร้างกราฟอันดับจังหวัดตาม count_want_land

เลือก 20 จังหวัดแรกเพื่อให้ชื่อจังหวัดอ่านง่าย และยังคงมี hover สำหรับดูค่าที่แน่นอน

In [11]:
top_demand_df = demand_supply_df.nlargest(20, WANT_COUNT_COLUMN).sort_values(
    WANT_COUNT_COLUMN
)
demand_ranking_figure = px.bar(
    top_demand_df,
    x=WANT_COUNT_COLUMN,
    y=PROVINCE_COLUMN,
    orientation="h",
    title="20 จังหวัดแรกตาม count_want_land",
    labels={WANT_COUNT_COLUMN: "ค่าใน column count_want_land", PROVINCE_COLUMN: "จังหวัด"},
    color_discrete_sequence=["#18794e"],
)
demand_ranking_figure.update_layout(template="plotly_white", height=700)
apply_plotly_font(demand_ranking_figure)

## สร้างกราฟอันดับจังหวัดตาม count_owner_land

เลือก 20 จังหวัดแรกเพื่อให้เปรียบเทียบค่าจาก column ที่พบจริงได้ชัดเจน

In [12]:
top_supply_df = demand_supply_df.nlargest(20, OWNER_LAND_COUNT_COLUMN).sort_values(
    OWNER_LAND_COUNT_COLUMN
)
supply_ranking_figure = px.bar(
    top_supply_df,
    x=OWNER_LAND_COUNT_COLUMN,
    y=PROVINCE_COLUMN,
    orientation="h",
    title="20 จังหวัดแรกตาม count_owner_land",
    labels={OWNER_LAND_COUNT_COLUMN: "ค่าใน column count_owner_land", PROVINCE_COLUMN: "จังหวัด"},
    color_discrete_sequence=["#1d4ed8"],
)
supply_ranking_figure.update_layout(template="plotly_white", height=700)
apply_plotly_font(supply_ranking_figure)

## สร้างกราฟส่วนต่างของ count fields ตามจังหวัด

กราฟนี้ใช้ครบ 77 จังหวัด และเรียงตาม `count_want_land - count_owner_land`

In [13]:
gap_by_province_df = demand_supply_df.sort_values(GAP_COLUMN)
gap_figure = px.bar(
    gap_by_province_df,
    x=GAP_COLUMN,
    y=PROVINCE_COLUMN,
    orientation="h",
    title="ส่วนต่าง count_want_land - count_owner_land ของ 77 จังหวัด",
    labels={GAP_COLUMN: "ส่วนต่างของ count fields", PROVINCE_COLUMN: "จังหวัด"},
    color=GAP_COLUMN,
    color_continuous_scale="RdBu_r",
)
gap_figure.update_layout(template="plotly_white", height=1800, coloraxis_showscale=False)
apply_plotly_font(gap_figure)

## สร้างกราฟกระจายระหว่าง demand และ supply

กราฟใช้ค่า `count_want_land` และ `count_owner_land` ของ 77 แถว พร้อมชื่อจังหวัดใน hover

In [14]:
scatter_figure = px.scatter(
    demand_supply_df,
    x=WANT_COUNT_COLUMN,
    y=OWNER_LAND_COUNT_COLUMN,
    hover_name=PROVINCE_COLUMN,
    title="ความสัมพันธ์ระหว่าง count_want_land และ count_owner_land",
    labels={
        WANT_COUNT_COLUMN: "count_want_land",
        OWNER_LAND_COUNT_COLUMN: "count_owner_land",
    },
    color_discrete_sequence=["#7c3aed"],
)
scatter_figure.update_layout(template="plotly_white", height=650)
apply_plotly_font(scatter_figure)

## เลือกแถวการจับคู่ที่มีค่าตัวเลข

ค่า `-` ถูกเก็บเป็น missing value ใน `จับคู่ได้_numeric` จึงไม่ถูกใช้ในกราฟเปรียบเทียบ

In [15]:
observed_matching_df = matching_df.dropna(subset=[MATCHED_NUMERIC_COLUMN]).copy()
observed_matching_df[
    [
        PROVINCE_COLUMN,
        "จับคู่ได้_numeric",
        "ผู้ประสงค์_numeric",
        "เจ้าของที่ดิน_numeric",
        "แปลงที่ดิน_numeric",
    ]
]

,province_name,จับคู่ได้_numeric,ผู้ประสงค์_numeric,เจ้าของที่ดิน_numeric,แปลงที่ดิน_numeric
0,กรุงเทพมหานคร,1.0,1.0,1.0,1.0
3,ปทุมธานี,1.0,1.0,1.0,1.0
7,สิงห์บุรี,1.0,1.0,1.0,1.0
16,นครนายก,1.0,3.0,1.0,1.0
37,เชียงใหม่,2.0,2.0,2.0,2.0
44,เชียงราย,1.0,11.0,1.0,1.0
57,นครปฐม,2.0,3.0,2.0,2.0


## สร้างกราฟเปรียบเทียบข้อมูลการจับคู่ที่มีตัวเลข

กราฟนี้แสดงเฉพาะ 7 แถวที่ `จับคู่ได้_numeric` มีค่า และไม่ใช้คำนวณอัตราการจับคู่

In [16]:
matching_long_df = observed_matching_df.melt(
    id_vars=[PROVINCE_COLUMN],
    value_vars=[
        "จับคู่ได้_numeric",
        "ผู้ประสงค์_numeric",
        "เจ้าของที่ดิน_numeric",
        "แปลงที่ดิน_numeric",
    ],
    var_name="column_name",
    value_name="observed_value",
)
matching_label_map = {
    "จับคู่ได้_numeric": "จับคู่ได้",
    "ผู้ประสงค์_numeric": "ผู้ประสงค์",
    "เจ้าของที่ดิน_numeric": "เจ้าของที่ดิน",
    "แปลงที่ดิน_numeric": "แปลงที่ดิน",
}
matching_long_df["column_label"] = matching_long_df["column_name"].replace(matching_label_map)

matching_figure = px.bar(
    matching_long_df,
    x=PROVINCE_COLUMN,
    y="observed_value",
    color="column_label",
    barmode="group",
    title="เปรียบเทียบค่าการจับคู่เฉพาะจังหวัดที่มีตัวเลข",
    labels={
        PROVINCE_COLUMN: "จังหวัด",
        "observed_value": "ค่าใน column ต้นฉบับ",
        "column_label": "column",
    },
)
matching_figure.update_layout(template="plotly_white", height=650)
apply_plotly_font(matching_figure)

## แปลงกราฟเป็น HTML fragments สำหรับ CDN

Fragment แรกโหลด Plotly จาก CDN เพียงครั้งเดียว ส่วน fragment ที่เหลือไม่โหลด JavaScript ซ้ำ

In [17]:
demand_ranking_html = pio.to_html(
    demand_ranking_figure,
    full_html=False,
    include_plotlyjs="cdn",
)
supply_ranking_html = pio.to_html(
    supply_ranking_figure,
    full_html=False,
    include_plotlyjs=False,
)
gap_html = pio.to_html(gap_figure, full_html=False, include_plotlyjs=False)
scatter_html = pio.to_html(scatter_figure, full_html=False, include_plotlyjs=False)
matching_html = pio.to_html(matching_figure, full_html=False, include_plotlyjs=False)

## แปลงกราฟแรกเป็น HTML fragment สำหรับ offline

fragment นี้ฝัง Plotly JavaScript ใน HTML เพื่อให้รายงาน offline ไม่ต้องโหลด JavaScript จาก CDN

In [18]:
demand_ranking_offline_html = pio.to_html(
    demand_ranking_figure,
    full_html=False,
    include_plotlyjs=True,
)

## สร้าง metric cards สำหรับ HTML

card แสดงตัวเลขที่คำนวณจาก cleaned output ของรายงานนี้เท่านั้น

In [19]:
metric_cards_html = "".join(
    [
        f"<article class='metric-card'><p>{escape(str(row.metric))}</p><strong>{escape(str(row.value))}</strong></article>"
        for row in key_metrics_df.itertuples(index=False)
    ]
)

## สร้าง HTML template ของรายงาน

เนื้อหาสรุปอ้างอิงผล Phase 3 และระบุข้อจำกัดไว้ในรายงานเพื่อไม่ให้ตีความเกินข้อมูล

In [20]:
html_report = f"""<!DOCTYPE html>
<html lang='th'>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1'>
  <title>รายงาน interactive LABAI Phase 3</title>
  <style>
    :root {{ color-scheme: light; }}
    @font-face {{ font-family: "Sarabun"; src: url("../../Sarabun-Regular.ttf") format("truetype"); font-style: normal; font-weight: 400; font-display: swap; }}
    body, h1, h2, h3, p, li, table, th, td, .metric-card, .chart-intro, .chart-note, .limitation, footer {{ font-family: "Sarabun", system-ui, sans-serif; }}
    body {{ margin: 0; background: #f5f7f8; color: #1f2933; font-family: "Sarabun", system-ui, sans-serif; line-height: 1.6; }}
    main {{ max-width: 1240px; margin: 0 auto; padding: 32px 24px 56px; }}
    header {{ border-bottom: 3px solid #0f766e; padding-bottom: 20px; }}
    h1, h2 {{ color: #123b42; }}
    h1 {{ margin: 0; font-size: 30px; }}
    h2 {{ margin-top: 40px; font-size: 22px; border-bottom: 1px solid #d8e2e5; padding-bottom: 8px; }}
    p {{ max-width: 1050px; }}
    .source-list, .note-list {{ padding-left: 20px; }}
    .source-list li, .note-list li {{ margin: 8px 0; }}
    .metric-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(205px, 1fr)); gap: 12px; }}
    .metric-card {{ background: #ffffff; border: 1px solid #d8e2e5; border-top: 4px solid #0f766e; padding: 14px; min-height: 98px; }}
    .metric-card p {{ margin: 0 0 8px; color: #52616b; font-size: 14px; }}
    .metric-card strong {{ font-size: 24px; color: #0f3f45; word-break: break-word; }}
    .chart-section {{ background: #ffffff; border: 1px solid #d8e2e5; margin: 22px 0; padding: 20px; }}
    .chart-section h3 {{ margin-top: 0; color: #123b42; }}
    .chart-intro {{ color: #52616b; }}
    .chart-note {{ border-left: 4px solid #0f766e; background: #effaf8; padding: 12px 14px; }}
    .limitation {{ border-left: 4px solid #b45309; background: #fffbeb; padding: 12px 14px; }}
    .idea-table {{ width: 100%; border-collapse: collapse; background: #ffffff; }}
    .idea-table th, .idea-table td {{ border: 1px solid #d8e2e5; padding: 10px; text-align: left; vertical-align: top; }}
    .idea-table th {{ background: #eaf4f3; color: #123b42; }}
    a {{ color: #0f5c75; overflow-wrap: anywhere; }}
    footer {{ border-top: 1px solid #cbd5e1; margin-top: 40px; padding-top: 16px; color: #52616b; font-size: 14px; }}
  </style>
</head>
<body>
  <main>
    <header>
      <h1>รายงาน interactive: ข้อมูลโครงการตัวกลางที่ดิน LABAI</h1>
      <p>รายงานนี้ใช้ข้อมูล cleaned ระดับจังหวัดจาก Phase 3 เพื่อทบทวนความสัมพันธ์ของ count fields, ส่วนต่างเชิงคำนวณ และข้อมูลการจับคู่ที่มีค่าตัวเลข โดยยังไม่มีการสร้าง model</p>
    </header>

    <section>
      <h2>วัตถุประสงค์</h2>
      <p>สนับสนุนการทบทวนข้อมูลก่อน Phase 4 ด้วยกราฟ interactive ที่ช่วยเปรียบเทียบจังหวัดและตรวจค่าใน column ที่พบจริง ผลทั้งหมดอธิบายเฉพาะ 77 แถวที่ดาวน์โหลดเมื่อ 2026-07-10</p>
    </section>

    <section>
      <h2>แหล่งข้อมูล</h2>
      <ul class='source-list'>
        <li>ข้อมูลผู้ประสงค์และเจ้าของที่ดินลงทะเบียน: <a href='https://data.go.th/dataset/65_dataset_11_03'>https://data.go.th/dataset/65_dataset_11_03</a></li>
        <li>ข้อมูลการจับคู่ที่ดิน: <a href='https://data.go.th/dataset/65_dataset_21_01'>https://data.go.th/dataset/65_dataset_21_01</a></li>
        <li>ข้อมูลเจ้าของที่ดินลงทะเบียน: <a href='https://data.go.th/dataset/65_dataset_11_01'>https://data.go.th/dataset/65_dataset_11_01</a></li>
      </ul>
    </section>

    <section>
      <h2>การตัดสินใจทำความสะอาด</h2>
      <ul class='note-list'>
        <li>ตัดช่องว่างต้นและท้ายของชื่อจังหวัดเป็น `province_name` โดยเก็บ column ต้นฉบับไว้</li>
        <li>area fields ที่เป็นข้อความถูกแปลงเป็น column ลงท้าย `_numeric` โดยยังไม่รวมหน่วย</li>
        <li>ค่า `-` ในข้อมูลการจับคู่ถูกเก็บเป็น missing value ใน column `_numeric` พร้อม flag `_has_placeholder`</li>
        <li>ไม่แทน `-` ด้วยศูนย์ เพราะ metadata ที่มีอยู่ไม่ยืนยันความหมายของสัญลักษณ์นี้</li>
      </ul>
    </section>

    <section>
      <h2>ตัวชี้วัดหลัก</h2>
      <div class='metric-grid'>{metric_cards_html}</div>
    </section>

    <section class='chart-section'>
      <h3>อันดับจังหวัดตาม count_want_land</h3>
      <p class='chart-intro'>กราฟนี้จัดอันดับ 20 จังหวัดแรกตามค่า `count_want_land` เพื่อช่วยตรวจการกระจุกตัวของค่าในไฟล์ cleaned demand/supply</p>
      {demand_ranking_html}
      <p class='chart-note'>สรุปการอ่าน: ใช้ `province_name` และ `count_want_land` หลังตัดช่องว่างชื่อจังหวัด ผลใช้ชี้จังหวัดที่ควรตรวจ metadata ต่อ ไม่ใช่ความต้องการที่ดินทั้งหมดของประเทศ</p>
    </section>

    <section class='chart-section'>
      <h3>อันดับจังหวัดตาม count_owner_land</h3>
      <p class='chart-intro'>กราฟนี้จัดอันดับ 20 จังหวัดแรกตามค่า `count_owner_land` เพื่อทบทวนการกระจายของ column นี้ในชุดข้อมูลเดียวกัน</p>
      {supply_ranking_html}
      <p class='chart-note'>สรุปการอ่าน: ค่าในกราฟต้องตีความตาม metadata และไม่ควรสรุปว่าเป็นจำนวนที่ดินทั้งหมดหรือความพร้อมของที่ดินในเชิงนโยบาย</p>
    </section>

    <section class='chart-section'>
      <h3>ส่วนต่างของ count fields</h3>
      <p class='chart-intro'>กราฟนี้เรียง 77 จังหวัดตามสูตร `count_want_land - count_owner_land` เพื่อช่วยตรวจค่าที่ต่างกันมากในข้อมูลที่ดาวน์โหลด</p>
      {gap_html}
      <p class='chart-note'>สรุปการอ่าน: ส่วนต่างเป็นเพียงค่าคำนวณจากชื่อ column ที่พบจริง ส่วนต่างบวกไม่ยืนยันภาวะขาดแคลนจนกว่าจะยืนยันนิยาม หน่วย และช่วงเวลาของข้อมูล</p>
    </section>

    <section class='chart-section'>
      <h3>กราฟกระจาย demand และ supply</h3>
      <p class='chart-intro'>กราฟนี้วางค่า `count_want_land` และ `count_owner_land` ของทั้ง 77 แถวเพื่อให้ตรวจรูปแบบร่วมและค่าที่อยู่ห่างจากกลุ่ม</p>
      {scatter_html}
      <p class='chart-note'>สรุปการอ่าน: ค่า correlation ใช้อธิบายความสัมพันธ์เชิงเส้นของข้อมูลชุดนี้เท่านั้น และไม่แสดงเหตุและผล</p>
    </section>

    <section class='chart-section'>
      <h3>ข้อมูลการจับคู่ที่มีค่าตัวเลข</h3>
      <p class='chart-intro'>กราฟนี้แสดงเฉพาะ 7 แถวที่ `จับคู่ได้_numeric` มีค่า เพื่อหลีกเลี่ยงการตีความ `-` เป็นศูนย์</p>
      {matching_html}
      <p class='chart-note'>สรุปการอ่าน: กราฟเปรียบเทียบค่าที่สังเกตได้เท่านั้น จึงไม่ใช้คำนวณอัตราการจับคู่หรือแทนค่า `-` ด้วยศูนย์</p>
    </section>

    <section>
      <h2>{IDEA_GALLERY_TITLE}</h2>
      <p>แนวคิดต่อไปนี้เป็นโอกาสต่อยอดจากแหล่งข้อมูล LABAI, SACIT และ KPI ยังไม่ได้ถูกนำมาใช้ในผลวิเคราะห์ LABAI ปัจจุบัน</p>
      <table class='idea-table'>
        <thead><tr><th>หน่วยงาน</th><th>แนวคิด</th><th>การต่อยอดที่เป็นไปได้</th></tr></thead>
        <tbody>
          <tr><td>LABAI</td><td>Land matching funnel</td><td>ตรวจขั้นตอนจากผู้ประสงค์, เจ้าของที่ดิน และการจับคู่ หลังยืนยันความหมายของ placeholder</td></tr>
          <tr><td>LABAI</td><td>Land vulnerability map</td><td>เปรียบเทียบจังหวัดจาก count และ area fields หลังยืนยันหน่วยข้อมูล</td></tr>
          <tr><td>SACIT</td><td>Craft export forecasting</td><td>วิเคราะห์แนวโน้มและ baseline forecast เมื่อมีช่วงเวลาที่ต่อเนื่องและครบถ้วน</td></tr>
          <tr><td>SACIT</td><td>Craft community map</td><td>สรุปชุมชนหัตถกรรมตามจังหวัด ภูมิภาค และประเภทข้อมูลที่ตรวจ schema แล้ว</td></tr>
          <tr><td>KPI</td><td>Governance, conflict and peace analytics</td><td>สร้าง dashboard หรือแผนที่เมื่อยืนยันเงื่อนไขเข้าถึงและรูปแบบข้อมูล</td></tr>
        </tbody>
      </table>
    </section>

    <section>
      <h2>ข้อจำกัดของข้อมูล</h2>
      <div class='limitation'>ไม่มี column เวลาในไฟล์ที่ใช้, ไม่ทราบความหมายของ `-` ในข้อมูลการจับคู่, และข้อมูลระดับจังหวัดมีเพียง 77 แถว area fields ยังต้องยืนยันหน่วยก่อนนำไปวิเคราะห์เพิ่มเติม</div>
    </section>

    <section>
      <h2>ข้อเสนอสำหรับ Phase 4</h2>
      <ul class='note-list'>
        <li>ยืนยันนิยามและหน่วยของ count และ area fields รวมถึงความหมายของ `-` ก่อนเลือก target</li>
        <li>เริ่ม regression baseline ได้เมื่อกำหนด target และตรวจความเหมาะสมของ feature</li>
        <li>classification ควรระบุชัดเจนว่า label เป็นกติกาที่สร้างขึ้น ไม่ใช่ label ต้นทาง</li>
        <li>clustering ต้อง standardize ข้อมูลและตรวจ outlier ก่อนประเมินด้วย silhouette score</li>
      </ul>
    </section>

    <footer>รายงานใช้ Sarabun จาก `../../Sarabun-Regular.ttf` เมื่อเบราว์เซอร์เข้าถึงไฟล์ได้ หากเข้าถึงไม่ได้ รายงานจะใช้ system font เป็น fallback และยังคงอ่านเนื้อหาและกราฟได้</footer>
  </main>
</body>
</html>
"""

## บันทึกรายงาน HTML แบบ CDN

ไฟล์นี้รวม chart fragments ไว้ในเอกสาร HTML เดียว และโหลด Plotly JavaScript จาก CDN เพียงครั้งเดียว

In [21]:
REPORT_FILE.write_text(html_report, encoding="utf-8")

65922

## สร้างรายงาน HTML แบบ offline

รายงานนี้แทน fragment กราฟแรกด้วยรุ่นที่ฝัง Plotly JavaScript และใช้ fragment ที่เหลือโดยไม่ฝัง JavaScript ซ้ำ

In [22]:
offline_html_report = html_report.replace(demand_ranking_html, demand_ranking_offline_html)
offline_html_report = offline_html_report.replace(
    "ใช้ Plotly JavaScript จาก CDN เพื่อให้ไฟล์ HTML มีขนาดเล็ก",
    "ฝัง Plotly JavaScript ในไฟล์เพื่อรองรับการเปิดแบบ offline",
)
OFFLINE_REPORT_FILE.write_text(offline_html_report, encoding="utf-8")

4916889

## ตรวจโครงสร้างของรายงาน offline

cell นี้ตรวจว่า HTML มีคำสั่งสร้างกราฟ, runtime ของ Plotly และชื่อหัวข้อกราฟครบทั้ง 5 รายการ
การตรวจนี้เป็นการตรวจเนื้อหาไฟล์ ไม่ใช่การตีความผลวิเคราะห์

In [23]:
offline_report_text = OFFLINE_REPORT_FILE.read_text(encoding="utf-8")
offline_chart_titles = [
    "อันดับจังหวัดตาม count_want_land",
    "อันดับจังหวัดตาม count_owner_land",
    "ส่วนต่างของ count fields",
    "กราฟกระจาย demand และ supply",
    "ข้อมูลการจับคู่ที่มีค่าตัวเลข",
]
offline_chart_title_checks = []

for chart_title in offline_chart_titles:
    offline_chart_title_checks.append(chart_title in offline_report_text)

offline_validation_results = {
    "has_plotly_newplot": "Plotly.newPlot" in offline_report_text,
    "has_plotly_runtime": "plotly.js" in offline_report_text or "PlotlyConfig" in offline_report_text,
    "has_all_chart_titles": all(offline_chart_title_checks),
    "has_five_chart_containers": offline_report_text.count('class="plotly-graph-div"') == 5,
    "has_no_external_cdn_script": '<script charset="utf-8" src="https://cdn.plot.ly/' not in offline_report_text,
    "has_idea_gallery": IDEA_GALLERY_TITLE in offline_report_text,
    "has_sarabun_font_references": PLOTLY_FONT_FAMILY in offline_report_text
    and "../../Sarabun-Regular.ttf" in offline_report_text,
    "has_font_fallback_note": "system font เป็น fallback" in offline_report_text,
}

offline_validation_df = pd.DataFrame(
    [
        {
            "has_plotly_newplot": offline_validation_results["has_plotly_newplot"],
            "has_plotly_runtime": offline_validation_results["has_plotly_runtime"],
            "has_all_chart_titles": offline_validation_results["has_all_chart_titles"],
            "has_five_chart_containers": offline_validation_results["has_five_chart_containers"],
            "has_no_external_cdn_script": offline_validation_results["has_no_external_cdn_script"],
            "has_idea_gallery": offline_validation_results["has_idea_gallery"],
            "has_sarabun_font_references": offline_validation_results["has_sarabun_font_references"],
            "has_font_fallback_note": offline_validation_results["has_font_fallback_note"],
        }
    ]
)
offline_validation_df

,has_plotly_newplot,has_plotly_runtime,has_all_chart_titles,has_five_chart_containers,has_no_external_cdn_script,has_idea_gallery,has_sarabun_font_references,has_font_fallback_note
0,True,True,True,True,True,True,True,True


## ยืนยันผลการตรวจ offline

หากตรวจไม่ผ่าน notebook จะหยุดเพื่อไม่ให้ส่งรายงาน offline ที่โครงสร้างไม่ครบ

In [24]:
if not all(offline_validation_results.values()):
    raise ValueError("รายงาน offline ไม่มีองค์ประกอบ Plotly หรือหัวข้อกราฟครบตามที่กำหนด")

## แหล่งข้อมูลและข้อจำกัด

ดูรายละเอียดแหล่งข้อมูลและข้อจำกัดได้ที่ `docs/data_sources.md` และ `docs/limitations.md`
